# **Scraping Planos Algar**

_Página do provedor: https://loja.algar.com.br/_

In [1]:
# Bibliotecas
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import date

## **1. Extração**

### **1. Extração do HTML**

In [ ]:
# Obetendo HTML
url = "https://loja.algar.com.br/"
response = requests.get(url)
html = response.text

# Exportando HTML
with open("../../data/scraped/algar.html", "w", encoding="utf-8") as f:
            f.write(html)

# Exibindo HTML
html

### **1.2 Scraping do HTML**

In [6]:
# Obtendo lista de planos
texto_html = open("../../data/scraped/algar.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
container_planos = soup.find_all("div", class_="swiper-wrapper")[1]
planos = container_planos.find_all("div", class_="swiper-slide")
len(planos)

7

In [7]:
# Raspando os planos
def raspar_plano(plano):
    try:
        velocidade = plano.find("h3").text.strip()
        preco = plano.find("div", class_="offerTile__price__main").text.strip()
        svas_tag = plano.find("div", class_="offerTile__offerSecondary")
        svas = svas_tag.text.strip() if svas_tag else None

        return {
            "velocidade": velocidade,
            "preco": preco,
            "svas": svas
        }
    except:
        return None

# Iterando sobre a lista de planos
dados = []
for plano in planos:
    item = raspar_plano(plano)
    if item:
        dados.append(item)

df = pd.DataFrame(dados)
df

,velocidade,preco,svas
0,600 Mega,"R$ 99,90\n \n /mês",NaN
1,800 Mega,"R$ 119,90\n \n /mês",NaN
2,600 Mega,"R$ 114,80\n \n /mês",+\n Globoplay com Anúncios
3,800 Mega,"R$ 134,80\n \n /mês",+\n Globoplay com Anúncios
4,600 Mega,"R$ 129,90\n \n /mês",+\n 01 Super Wi-Fi
5,800 Mega,"R$ 149,90\n \n /mês",+\n 01 Super Wi-Fi
6,600 Mega,"R$ 149,90\n \n /mês",+\n Celular Controle Pl...


## **2. Tratamento**

In [ ]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Algar"

# Velocidade: captura a priemeira sequência de números inteiros da string
df_tratado["VELOCIDADE"] = df_tratado["velocidade"].str.extract(r"(\d+)").astype(float)

# SVAs: remove "+\n" e remove espaços espaços em excesso
df_tratado["SVAs"] = (
    df_tratado["svas"]
    .str.replace(r"\+\n", "", regex=True)
    .str.strip()
)

# Preço: captura um seguencia de inteiros + vírgula + sequencia de inteiros
# Substui vírgula por ponto
# Convert string em float
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .str.extract(r"(\d+,\d+)", expand=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

,DATA,ISP,VELOCIDADE,SVAs,PRECO
0,2026-04-20,Algar,600.0,NaN,99.9
1,2026-04-20,Algar,800.0,NaN,119.9
2,2026-04-20,Algar,600.0,Globoplay com Anúncios,114.8
3,2026-04-20,Algar,800.0,Globoplay com Anúncios,134.8
4,2026-04-20,Algar,600.0,01 Super Wi-Fi,129.9
5,2026-04-20,Algar,800.0,01 Super Wi-Fi,149.9
6,2026-04-20,Algar,600.0,Celular Controle Plus 55 Giga,149.9


## **3. Carga**

In [9]:
# Salvando
nome_arq = df_tratado["DATA"].max() + "-ALGAR.csv"

#pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
#df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

df_tratado.to_csv(f"../../data/flat/{nome_arq}", index=False)
print(f"🟢 {nome_arq} salvo!")

🟢 2026-04-20-ALGAR.csv salvo!
